# Homework 4 — Operational Space Control

### Purpose

One of the goals of this course is to:

> **Put the robot arm's end-effector at a commanded position *and* orientation.**

So far you have reached that goal with *joint angles* (IK). Real arms are driven by
**torques**, and a torque command has to fight the arm's own inertia, its Coriolis
and centrifugal forces, and gravity. In this homework you will learn how to compute the torques that produce a desired motion.

The idea of **operational space control** (OSC) is to write Newton's law directly in
task coordinates,

$$\Lambda(q)\,\ddot x + \mu(q,\dot q) + \rho(q) = F, \qquad \tau = J^{\top}F,$$

design the controller as if the tool were a free body with apparent inertia
$\Lambda$, and convert the task force $F$ to joint torques with $J^{\top}$.

**Overview.**

1. **Q1 Dynamics** — get $M$, $C$, $G$ from MuJoCo and build $\Lambda$, $\mu$, $\rho$.
2. **Q2 Baseline** — inverse kinematics plus a joint-space PD.
3. **Q3 OSC** — implement the control law and track a circle. (On a 7-joint arm it fails, and you will find out why.)
4. **Q4 Null space** — fix it with the seventh joint.
5. **Q5 Variants** — remove one term at a time and measure what each one does.
6. **Q6 Gain tuning** — under OSC the tool error is a mass-spring-damper; check it against second-order theory.
7. **Q7 Control rate (bonus)** — how far the gains can go at a finite control rate.

**Grading:** 100 points (Q1–Q6), plus 10 bonus points (Q7).
$\color{red}{\text{Use numpy, matplotlib, mujoco only in your code.}}$
(The given rendering helpers also use `imageio` and IPython.)

### Notation and conventions

**Joint-space dynamics.** For an arm with joint angles $q \in \mathbb{R}^7$,

$$M(q)\,\ddot q + C(q,\dot q) + G(q) = \tau$$

* $M(q)$ — $7\times7$ joint-space inertia matrix (symmetric, positive definite)
* $C(q,\dot q)$ — Coriolis and centrifugal torques; quadratic in $\dot q$, zero at rest
* $G(q)$ — gravity torques
* $\tau$ — joint torques from the motors

You will derive these from a robot's geometry and mass properties later in the
course. In this homework MuJoCo computes them; your job is to use them.

**Task space.** $x$ is the tool pose, $\dot x = J\dot q$ is its twist and
$\ddot x = J\ddot q + \dot J\dot q$ its acceleration. $J$ is the $6\times7$ world
Jacobian of the site `ee_tip`. Twists, accelerations and forces are 6-vectors ordered
$[\,\omega\,;\,v\,]$ — **angular first**. The pose error is

$$e = \begin{bmatrix}\log(R_{des}R^{\top})^{\vee} \\ p_{des}-p\end{bmatrix},$$

where only the position half is a plain subtraction.

**The model.** `Forte.xml` sets `armature="0.01"` on every joint (reflected rotor
inertia of a geared motor). Joint damping and friction are
**zero**. Each motor has a torque limit (`TAU_LIM`); commands are clipped
to it.

**Joint limits.** The pitch joints rotate about $+y$, so a positive angle swings the
link **downward**: negative `shoulder_pitch` lifts the arm, and `elbow_pitch` is $0$
when fully stretched and positive as the forearm folds in. Shoulder roll, shoulder pitch and
wrist pitch stop at $\pm90^\circ$, the elbow at $[0, 180^\circ]$, and the other joints at $\pm180^\circ$.

In [ ]:
# No need to change
import numpy as np
import matplotlib.pyplot as plt
import mujoco

np.set_printoptions(precision=4, suppress=True)

MODEL = "../ForteArm_Model/forte_scene.xml"
m = mujoco.MjModel.from_xml_path(MODEL)
d = mujoco.MjData(m)
NV = m.nv                                    # 7 joints
DT = m.opt.timestep                          # 2 ms
EE = mujoco.mj_name2id(m, mujoco.mjtObj.mjOBJ_SITE, "ee_tip")
TAU_LIM = m.actuator_ctrlrange[:, 1].copy()  # motor torque limits [Nm]
Q_LO = m.jnt_range[:, 0].copy()              # joint travel limits [rad]
Q_HI = m.jnt_range[:, 1].copy()
KEY_READY = 1                                # keyframe index of `ready_pose`

def reset():
    """Put the arm at rest at `ready_pose`."""
    mujoco.mj_resetDataKeyframe(m, d, KEY_READY)
    mujoco.mj_forward(m, d)

def log_SO3(R):
    """Rotation matrix -> axis*angle 3-vector."""
    c = np.clip((np.trace(R) - 1) / 2.0, -1.0, 1.0)
    if c > 1 - 1e-12:                        # tiny angle: log(R) ~ vee((R - R^T)/2)
        S = (R - R.T) / 2.0
        return np.array([S[2, 1], S[0, 2], S[1, 0]])
    if c < -1 + 1e-12:                       # angle = pi
        A = R + np.eye(3); k = int(np.argmax(np.linalg.norm(A, axis=0)))
        return np.pi * A[:, k] / np.linalg.norm(A[:, k])
    th = np.arccos(c); S = (R - R.T) / (2 * np.sin(th))
    return th * np.array([S[2, 1], S[0, 2], S[1, 0]])

def tool_pose():
    """Current tool position (3,) and orientation (3x3) in the world frame."""
    return d.site_xpos[EE].copy(), d.site_xmat[EE].reshape(3, 3).copy()

def pose_error(p_des, R_des):
    """6-D pose error [omega; v]."""
    p, R = tool_pose()
    return np.concatenate([log_SO3(R_des @ R.T), p_des - p])

def get_J():
    """6 x 7 world Jacobian of the tool, ordered [omega; v] (MuJoCo's mj_jacSite)."""
    Jp = np.zeros((3, NV)); Jr = np.zeros((3, NV))
    mujoco.mj_jacSite(m, d, Jp, Jr, EE)
    return np.vstack([Jr, Jp])

def get_Jdot_qdot():
    """Jdot * qdot: the tool acceleration when qddot = 0, ordered [omega; v].

    MuJoCo has no direct call for this. The tool acceleration it reports for
    qacc = 0 is exactly Jdot*qdot, except that mj_objectAcceleration folds
    gravity into the linear part, which we add back.
    """
    qacc_keep = d.qacc.copy()
    d.qacc[:] = 0
    mujoco.mj_rnePostConstraint(m, d)
    acc = np.zeros(6)
    mujoco.mj_objectAcceleration(m, d, mujoco.mjtObj.mjOBJ_SITE, EE, acc, 0)
    acc[3:] += m.opt.gravity
    d.qacc[:] = qacc_keep
    return acc

def check(name, got, want, tol=1e-8):
    ok = np.allclose(np.asarray(got, float), np.asarray(want, float), atol=tol)
    print(f'{"PASS" if ok else "FAIL"}  {name}')
    if not ok:
        print('   yours   :', np.asarray(got)); print('   expected:', np.asarray(want))
    return ok

reset()
p_home, R_home = tool_pose()                 # tool pose at ready_pose
q_home = d.qpos.copy()
print('model:', MODEL, '| nv =', NV, '| timestep =', DT, 's')
print('torque limits [Nm]:', TAU_LIM)
print('ready_pose q =', q_home, '\ntool at', p_home)

---
# Q1 [15 pts] The dynamics, in joint space and in task space

**(a) Joint space.** Complete `get_M`, `get_G` and `get_C`.

* `mujoco.mj_fullM` converts MuJoCo's internal (sparse) inertia into a dense $7\times7$
  matrix. Its arguments changed between MuJoCo versions: recent versions take
  `mj_fullM(m, d, M)`, older ones `mj_fullM(m, M, d.qM)`. Try the first, and fall back
  to the second if it raises a `TypeError`.
* MuJoCo stores $C + G$ in `d.qfrc_bias`. Separate the two: $C$ vanishes when
  $\dot q = 0$. Evaluate $G$ on the scratch copy `_dg`, so that you never change the
  state of the simulation you are controlling.

**(b) Task space.** Substituting $\ddot q = M^{-1}(\tau - C - G)$ into
$\ddot x = J\ddot q + \dot J\dot q$ with $\tau = J^{\top}F$ gives

$$\Lambda\,\ddot x + \mu + \rho = F, \qquad
\Lambda = \big(JM^{-1}J^{\top}\big)^{-1},\quad
\mu = \Lambda JM^{-1}C - \Lambda\dot J\dot q,\quad
\rho = \Lambda JM^{-1}G .$$

Complete `task_dynamics`. The self-test applies a random task force $F$ as
$\tau = J^{\top}F$, lets MuJoCo compute the resulting motion, and checks that your
$\Lambda$, $\mu$, $\rho$ reproduce $F$.

**(c)** $\Lambda$ is the tool's **apparent inertia**: the force needed to give the tool
a unit acceleration in each direction. The cell prints its eigenvalues at `ready_pose`
and at `q = 0` (arm straight out along $x$). Which eigenvalues belong to rotation and
which to translation (look at the units)? What happens to $\Lambda$ at `q = 0`, and
what does that mean for the force OSC will ask for there? 3–4 sentences.

In [ ]:
def get_M():
    """Joint-space inertia matrix M(q), NV x NV."""
    # TODO Q1a: fill the dense M with mujoco.mj_fullM. Recent MuJoCo: mj_fullM(m, d, M);
    #           older MuJoCo: mj_fullM(m, M, d.qM). Try one, fall back on TypeError.
    raise NotImplementedError("Q1: get_M")

_dg = mujoco.MjData(m)      # scratch copy: evaluating G must not disturb `d`

def get_G():
    """Gravity torque G(q), NV."""
    # TODO Q1a: d.qfrc_bias holds C + G. Copy the joint angles into the scratch data _dg,
    #           set its velocities to zero (so C = 0), run mj_forward on it, and read
    #           its qfrc_bias.
    raise NotImplementedError("Q1: get_G")

def get_C():
    """Coriolis + centrifugal torque C(q, qdot), NV."""
    # TODO Q1a: C is whatever is left of qfrc_bias once G is removed.
    raise NotImplementedError("Q1: get_C")

def task_dynamics():
    """Operational-space quantities at the current state: Lambda (6x6), mu (6), rho (6)."""
    M = get_M(); Minv = np.linalg.inv(M); J = get_J()
    # TODO Q1b: Lam = (J M^-1 J^T)^-1,  mu = Lam J M^-1 C - Lam Jdot qdot,
    #           rho = Lam J M^-1 G.  (get_C, get_G, get_Jdot_qdot are available.)
    raise NotImplementedError("Q1: task_dynamics")
    return Lam, mu, rho

In [ ]:
# --- self test ---
rng = np.random.default_rng(0)
reset()
qd = rng.normal(size=NV) * 0.5

# (a) joint space
d.qvel[:] = qd; mujoco.mj_forward(m, d)
M, C, G = get_M(), get_C(), get_G()
check('Q1 M symmetric', M, M.T, tol=1e-12)
check('Q1 M positive definite', np.linalg.eigvalsh(M).min() > 0, True)

d.qvel[:] = 0; mujoco.mj_forward(m, d)
check('Q1 C = 0 at rest', get_C(), np.zeros(NV), tol=1e-12)

d.qvel[:] = 2 * qd; mujoco.mj_forward(m, d)
check('Q1 C is quadratic in qdot: C(2 qdot) = 4 C(qdot)', get_C(), 4 * C, tol=1e-10)

tau_test = rng.normal(size=NV)
d.qvel[:] = qd; d.qfrc_applied[:] = tau_test    # apply a known joint torque ...
mujoco.mj_forward(m, d)                         # ... and let MuJoCo compute qddot
check('Q1 M qddot + C + G = tau', M @ d.qacc + C + G, tau_test, tol=1e-10)

# (b) task space: apply tau = J^T F and check Lambda xddot + mu + rho = F
F_test = rng.normal(size=6)
d.qfrc_applied[:] = get_J().T @ F_test
mujoco.mj_forward(m, d)
Lam, mu, rho = task_dynamics()
xdd = get_J() @ d.qacc + get_Jdot_qdot()
check('Q1 Lambda xddot + mu + rho = F', Lam @ xdd + mu + rho, F_test, tol=1e-8)
check('Q1 Lambda symmetric', Lam, Lam.T, tol=1e-8)
d.qfrc_applied[:] = 0

# (c) apparent inertia
for name, q in [('ready_pose', q_home), ('q = 0', np.zeros(NV))]:
    reset(); d.qpos[:] = q; mujoco.mj_forward(m, d) 
    Lam, _, _ = task_dynamics()
    print(f'\n{name:10s}  eig(Lambda) = {np.linalg.eigvalsh(Lam)}')
    print(f'{"":10s}  cond(Lambda) = {np.linalg.cond(Lam):9.1f}   cond(M) = {np.linalg.cond(get_M()):6.1f}'
          f'   sigma_min(J) = {np.linalg.svd(get_J(), compute_uv=False).min():.4f}')
reset()

**(Your answer for Q1c)**

*write here*

---
# Q2 [10 pts] Baseline: inverse kinematics + joint PD

The approach you already know: solve IK once for the target, then move every joint
to its IK angle,

$$\tau = K_p\,(q_{des}-q) - K_d\,\dot q\ \big[\, + G(q)\,\big].$$

The target is a **stationary** pose 12 cm from the tool's starting position.

**(a)** Complete `joint_pd`, with an option to add gravity compensation $G(q)$.

**(b)** Write the code for the two experiments in the cell below, using the given
`simulate(controller, T)`. A controller is a function `controller(t)` that returns
`(tau, p_ref, R_ref)`.

1. Run `joint_pd` toward `q_goal` with and without gravity compensation, and store the
   results as `res[True]` and `res[False]`. Run the case **without** $G$ last: the
   equilibrium check after it reads the arm's final state.
2. Re-run the uncompensated controller for 4 s with the gains scaled by
   $s \in \{1, 2, 4, 10\}$. Scale $K_p$ by $s$ and $K_d$ by $\sqrt{s}$, which keeps
   the damping ratio unchanged. Store the final tool position error as `sweep[s]`.

The cell then prints the errors and the equilibrium check, and plots both runs.

**(c)** Explain the steady-state error without $G$: write down the equilibrium
condition at rest and show it predicts the measured joint offset. Why does adding $G$
remove the error, while raising $K_p$ only shrinks it? Use the numbers from the gain
sweep.

In [ ]:
def joint_pd(q_des, Kp, Kd, gravity_comp):
    """Joint-space PD, optionally with gravity compensation. Returns a torque vector."""
    # TODO Q2a: joint-space PD on (q_des - q) and qdot; add G(q) if gravity_comp.
    raise NotImplementedError("Q2: joint_pd")

def simulate(controller, T=3.0):
    """Run controller(t) -> (tau, p_ref, R_ref) from ready_pose for T seconds.
    Returns time, position error [m] and orientation error [rad]."""
    reset()
    ts, pe, oe = [], [], []
    for k in range(int(T / DT)):
        t = k * DT
        tau, p_ref, R_ref = controller(t)
        d.ctrl[:] = np.clip(tau, -TAU_LIM, TAU_LIM)
        mujoco.mj_step(m, d)
        p, R = tool_pose()
        ts.append(t); pe.append(np.linalg.norm(p_ref - p))
        oe.append(np.linalg.norm(log_SO3(R_ref @ R.T)))
    return np.array(ts), np.array(pe), np.array(oe)

def ik(p_des, R_des, q0, iters=300, lam=1e-2):
    """Damped least-squares Newton IK, projected into the joint limits."""
    dd = mujoco.MjData(m); q = np.clip(q0.copy(), Q_LO, Q_HI)
    for _ in range(iters):
        dd.qpos[:] = q; mujoco.mj_forward(m, dd)
        p = dd.site_xpos[EE]; R = dd.site_xmat[EE].reshape(3, 3)
        e = np.concatenate([log_SO3(R_des @ R.T), p_des - p])
        if np.linalg.norm(e) < 1e-10:
            break
        Jp = np.zeros((3, NV)); Jr = np.zeros((3, NV)); mujoco.mj_jacSite(m, dd, Jp, Jr, EE)
        J = np.vstack([Jr, Jp])
        q = np.clip(q + J.T @ np.linalg.solve(J @ J.T + lam**2 * np.eye(6), e), Q_LO, Q_HI)
    return q

In [ ]:
p_goal = p_home + np.array([0.0, 0.10, 0.07])     # 12 cm away, same orientation
R_goal = R_home.copy()
q_goal = ik(p_goal, R_goal, q_home)
reset(); d.qpos[:] = q_goal; mujoco.mj_forward(m, d)
check('Q2 IK solved the target', np.linalg.norm(pose_error(p_goal, R_goal)), 0.0, tol=1e-6)

Kp_j = np.array([80, 120, 60, 60, 20, 12, 5.0])
Kd_j = 0.7 * np.sqrt(Kp_j)

res = {}       # res[gravity_comp] = (t, position error, orientation error)
# TODO Q2b-1: for gc in (True, False) -- WITHOUT G last -- store
#   res[gc] = simulate(controller) with controller(t) returning
#   (joint_pd(q_goal, Kp_j, Kd_j, gc), p_goal, R_goal).
#   Careful with a lambda inside a loop: bind gc as a default argument (gc=gc).
raise NotImplementedError("Q2: gravity experiment")
for gc in (False, True):
    print(f'gravity_comp = {str(gc):5s}: final tool error = {res[gc][1][-1]*1000:8.3f} mm')
check('Q2 gravity compensation removes the steady-state error', res[True][1][-1] < 1e-4, True)

# equilibrium check: at rest without G, the joint offset should be G / Kp
dq_rest = q_goal - d.qpos
print('\njoint offset at rest  q_des - q :', dq_rest)
print('predicted        G(q) / Kp     :', get_G() / Kp_j)

GAIN_SCALES = (1, 2, 4, 10)
sweep = {}     # sweep[s] = final tool position error [m] without G, gains scaled by s
# TODO Q2b-2: for s in GAIN_SCALES, simulate for T=4.0 s without G, with gains
#   s*Kp_j and sqrt(s)*Kd_j, and store the final position error as sweep[s].
raise NotImplementedError("Q2: gain sweep")
print('\nwithout G, raising the gains:')
for s in GAIN_SCALES:
    print(f'   {s:2d} x Kp : final tool error = {sweep[s]*1000:7.3f} mm')

plt.figure(figsize=(7, 4))
for gc, lab in [(False, 'joint PD'), (True, 'joint PD + G(q)')]:
    plt.plot(res[gc][0], res[gc][1] * 1000, label=lab)
plt.xlabel('t [s]'); plt.ylabel('tool position error [mm]'); plt.legend(); plt.grid(alpha=0.3)
plt.title('Q2: IK + joint-space PD'); plt.show()

In [ ]:
# replay: IK + joint PD toward the stationary target, without and with G (3 s each, real time)
# Rendering is given and not needed for any answer. On Linux without a display, set MUJOCO_GL=egl;
# if it still fails, skip this cell (and the other rendering cells) -- the numbers above are enough.
import sys; sys.path.append('../ForteArm_Model')   # so we can import the visualization code
from forte_viz import ForteViz, show_video
from IPython.display import display, Markdown
viz = ForteViz()
cam = viz.camera(zoom=1.5)
for k, (gc, lab) in enumerate([(False, 'joint PD'), (True, 'joint PD + G(q)')]):
    reset(); frames = []
    for step in range(int(3.0 / DT)):
        d.ctrl[:] = np.clip(joint_pd(q_goal, Kp_j, Kd_j, gc), -TAU_LIM, TAU_LIM)
        mujoco.mj_step(m, d)
        if step % 15 == 0:                     # a frame every 30 ms
            frames.append(viz.frame(d.qpos, (p_goal, R_goal), cam=cam))
    display(Markdown(f'**{lab}** (final tool error {res[gc][1][-1]*1000:.1f} mm)'))
    display(show_video(frames, fps=33, path=f'osc_q2_{k + 1}.mp4', width=600))

**(Your answer for Q2c)**

*write here*

---
# Q3 [20 pts] Operational space control

The controller from class: choose the tool acceleration you want,

$$\ddot x_d = \ddot x_{ff} + K_p\,(x_{ref} - x) + K_d\,(\dot x_{ref} - \dot x),$$

and compute the force that produces it from the task-space dynamics of Q1:

$$F = \Lambda\,\ddot x_d + \mu + \rho, \qquad \tau = J^{\top}F .$$

$\ddot x_{ff}$ is the feedforward acceleration of the reference trajectory.
For the orientation half, $x_{ref} - x$ is not a subtraction: it is the rotation
error $\log(R_{ref}R^{\top})^{\vee}$, which `pose_error` already computes.
$K_p$, $K_d$ are 6-vectors of gains, one per direction.

**(a) Reference trajectory.** Complete `circle(t)`. The tool goes around a circle of
radius $r$ = `RADIUS` = 0.10 m at $f$ = `FREQ` = 0.5 Hz in the $y$–$z$ plane, with its
orientation held fixed at `R_home`. The circle passes through the starting tool
position `p_home`; its center is `CENTER` $= p_{home} - r\,\hat y$ (10 cm toward $-y$).
With $\omega = 2\pi f$,

$$p_{ref}(t) = c + r\begin{bmatrix}0\\ \cos\omega t\\ \sin\omega t\end{bmatrix},$$

so $p_{ref}(0) = p_{home}$ and the tool starts by moving straight up ($+z$).

`circle(t)` returns three arrays:

| name | size | contents |
|---|---|---|
| `p_ref` | 3 | position $p_{ref}(t)$ [m] |
| `v_ref` | 6 | reference twist $[\,\omega\,;\,v\,]$: angular part $0$ (fixed orientation), linear part $\dot p_{ref}(t)$ [m/s] |
| `a_ff` | 6 | feedforward acceleration $[\,\dot\omega\,;\,\dot v\,]$: angular part $0$, linear part $\ddot p_{ref}(t)$ [m/s²] |

Differentiate $p_{ref}(t)$ by hand to get $\dot p_{ref}$ and $\ddot p_{ref}$. The
6-vectors are ordered **angular first**, as everywhere in this course. The test cell
checks that the path starts at `p_home`, stays on a circle of radius $r$ in a plane of
constant $x$, repeats every $1/f$ seconds, and starts upward at speed $r\omega$. It also
checks that your velocity and acceleration agree with finite differences of your
position, that the acceleration is centripetal ($-\omega^2(p_{ref} - c)$), and that the
angular parts are zero. Then it plots the path.

**(b)** Complete `osc_task`. Setting `use_mu`, `use_rho` or `use_ff` to `False` switches the
corresponding term off, and `use_Lambda=False` replaces the whole law by a plain
Cartesian PD through the Jacobian, $\tau = J^{\top}\ddot x_d + G$. Q5 uses them;
respect them.

**(c)** The cell after the test tracks your circle for 6 s with `osc_task`. It plots the
tool error and the joint angles against their limits. (The first 0.5 s is a start-up
transient: the reference starts moving while the arm is at rest. It is excluded from the
statistics.)

**(d)** For the first two seconds the error is about 0.6 mm. Then it falls
apart. Explain what goes wrong: which joints move, where they end up, and why the
tool error stays small until then. Why did the planar 3-joint example in class, with a
3-D task $(x, y, \theta)$, never show this? *Hint:* count the rows and columns of $J$.


In [ ]:
def osc_task(p_ref, R_ref, v_ref, a_ff, Kp, Kd,
             use_Lambda=True, use_mu=True, use_rho=True, use_ff=True):
    """tau = J^T [Lambda xdd_d + mu + rho]. Returns a joint torque vector."""
    J = get_J()
    e = pose_error(p_ref, R_ref)             # x_ref - x, ordered [omega; v]
    xdot = J @ d.qvel                         # current tool twist
    # TODO Q3b: the OSC law.
    #   xdd_d = xdd_ff + Kp (x_ref - x) + Kd (v_ref - xdot), with x_ref - x = e  (drop xdd_ff if not use_ff)
    #   if not use_Lambda: return J^T xdd_d + G         (plain Cartesian PD)
    #   F     = Lam xdd_d + mu + rho                    (respect use_mu / use_rho)
    #   tau   = J^T F
    raise NotImplementedError("Q3: osc_task")


In [ ]:
RADIUS, FREQ = 0.10, 0.5                      # 10 cm circle at 0.5 Hz in the y-z plane
CENTER = p_home - np.array([0.0, RADIUS, 0.0]) # the circle passes through p_home

def circle(t):
    """Reference for the tool at time t.
    Returns p_ref (3,), v_ref (6,) = [omega; v], a_ff (6,) = [omega_dot; v_dot]."""
    # TODO Q3a: with w = 2 pi FREQ, p_ref = CENTER + RADIUS * [0, cos(w t), sin(w t)].
    #   Differentiate it by hand for the linear velocity and acceleration.
    #   v_ref and a_ff are 6-vectors [angular; linear]; the angular parts are zero.
    raise NotImplementedError("Q3: circle")
    return p_ref, v_ref, a_ff


In [ ]:
# --- self test of circle(t) ---
w = 2 * np.pi * FREQ
p0, v0, a0 = circle(0.0)
check('Q3 circle returns sizes 3, 6, 6', [np.size(p0), np.size(v0), np.size(a0)], [3, 6, 6])
check('Q3 starts at p_home', p0, p_home, tol=1e-12)
check('Q3 starts moving straight up at speed r*omega', v0[3:], [0, 0, RADIUS * w], tol=1e-12)

ts_test = np.linspace(0, 1 / FREQ, 101)
traj = [circle(t) for t in ts_test]
P = np.array([p for p, _, _ in traj])
check('Q3 stays on the circle (|p - c| = r)', np.linalg.norm(P - CENTER, axis=1), RADIUS * np.ones(len(P)), tol=1e-12)
check('Q3 stays in the plane x = const', P[:, 0], p_home[0] * np.ones(len(P)), tol=1e-12)
check('Q3 periodic: circle(1/f) = circle(0)', np.r_[traj[-1][0], traj[-1][1], traj[-1][2]],
      np.r_[p0, v0, a0], tol=1e-9)
check('Q3 angular parts are zero (orientation fixed)',
      np.r_[[v[:3] for _, v, _ in traj], [a[:3] for _, _, a in traj]], np.zeros((2 * len(traj), 3)), tol=1e-15)

h = 1e-6                                          # central finite differences
fd_v = [(circle(t + h)[0] - circle(t - h)[0]) / (2 * h) for t in ts_test]
fd_a = [(circle(t + h)[1][3:] - circle(t - h)[1][3:]) / (2 * h) for t in ts_test]
check('Q3 v_ref = d p_ref / dt (finite difference)', [v[3:] for _, v, _ in traj], fd_v, tol=1e-6)
check('Q3 a_ff = d v_ref / dt (finite difference)', [a[3:] for _, _, a in traj], fd_a, tol=1e-5)
check('Q3 a_ff is centripetal: -omega^2 (p - c)', [a[3:] for _, _, a in traj], -w**2 * (P - CENTER), tol=1e-12)

fig, ax = plt.subplots(figsize=(5, 5))
ax.plot(P[:, 1], P[:, 2], 'k-', lw=1.5, label='p_ref(t), one period')
for p, v, a in traj[::10]:
    ax.arrow(p[1], p[2], 0.05 * v[4], 0.05 * v[5], head_width=0.004, color='tab:blue')
    ax.arrow(p[1], p[2], 0.02 * a[4], 0.02 * a[5], head_width=0.004, color='tab:red')
ax.plot(*p_home[1:], 'go', label='p_home (t = 0)'); ax.plot(*CENTER[1:], 'k+', ms=12, label='CENTER')
ax.plot([], [], color='tab:blue', label='velocity (scaled)'); ax.plot([], [], color='tab:red', label='acceleration (scaled)')
ax.set_xlabel('y [m]'); ax.set_ylabel('z [m]'); ax.set_aspect('equal'); ax.grid(alpha=0.3)
ax.legend(fontsize=8, loc='lower left'); ax.set_title('Q3(a): reference circle'); plt.show()


In [ ]:
Kp_t = np.array([100, 100, 100, 400, 400, 400.])   # [omega; v]
Kd_t = 2 * np.sqrt(Kp_t)                           # critically damped (see Q6)

def run_circle(T=4.0, Kp=Kp_t, Kd=Kd_t, null_space=None, decim=1, stride=0, **flags):
    """Track the circle from ready_pose.
    null_space : None, or a function returning the extra torque to add (Q4)
    decim      : recompute the torque every `decim` steps, hold it in between (Q7)
    stride     : if > 0, also keep every stride-th configuration for rendering
    Returns a dict of results, or None if the simulation blew up."""
    reset()
    tau = np.zeros(NV)
    ts, pe, oe, tmax, sat, qs, frames = [], [], [], [], [], [], []
    for k in range(int(T / DT)):
        t = k * DT
        p_ref, v_ref, a_ff = circle(t)
        if k % decim == 0:
            tau = osc_task(p_ref, R_home, v_ref, a_ff, Kp, Kd, **flags)
            if null_space is not None:
                tau = tau + null_space()
        d.ctrl[:] = np.clip(tau, -TAU_LIM, TAU_LIM)
        mujoco.mj_step(m, d)
        if not np.isfinite(d.qpos).all() or np.abs(d.qvel).max() > 1e3:
            return None
        p, R = tool_pose()
        ts.append(t); pe.append(np.linalg.norm(p_ref - p)); oe.append(np.linalg.norm(log_SO3(R_home @ R.T)))
        tmax.append(np.abs(tau).max()); sat.append(np.any(np.abs(tau) > TAU_LIM)); qs.append(d.qpos.copy())
        if stride and k % stride == 0:
            frames.append((d.qpos.copy(), (p_ref.copy(), R_home.copy())))
    ts, pe, oe, qs = map(np.array, (ts, pe, oe, qs))
    s = ts > 0.5
    margin = np.minimum(qs - Q_LO, Q_HI - qs).min()
    return dict(t=ts, pe=pe, oe=oe, q=qs, frames=frames,
                pos_med=np.median(pe[s]), pos_max=pe[s].max(), ori_med=np.median(oe[s]),
                tau_max=max(tmax), sat=100 * np.mean(sat),
                drift=np.linalg.norm(qs[-1] - q_home), margin=margin)

JOINTS = ['shoulder_yaw', 'shoulder_pitch', 'shoulder_roll', 'elbow_pitch',
          'lower_arm_roll', 'wrist_pitch', 'wrist_roll']

def plot_run(r, title):
    fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
    ax[0].semilogy(r['t'], r['pe'] * 1000 + 1e-4)
    ax[0].set_xlabel('t [s]'); ax[0].set_ylabel('tool position error [mm]')
    ax[0].grid(alpha=0.3, which='both'); ax[0].set_title(title)
    for j in range(NV):
        line, = ax[1].plot(r['t'], r['q'][:, j], label=JOINTS[j])
        for lim in (Q_LO[j], Q_HI[j]):
            if abs(lim) < 2:
                ax[1].axhline(lim, color=line.get_color(), ls=':', lw=1)
    ax[1].set_xlabel('t [s]'); ax[1].set_ylabel('joint angle [rad]')
    ax[1].set_title('joint angles (dotted: limits)'); ax[1].legend(fontsize=7, ncol=2)
    ax[1].grid(alpha=0.3); plt.tight_layout(); plt.show()

lec = run_circle(T=6.0, stride=10)
early = (lec['t'] > 0.5) & (lec['t'] < 2.0)
print(f'median error 0.5-2 s : {np.median(lec["pe"][early])*1000:7.3f} mm')
print(f'median error 2-6 s   : {np.median(lec["pe"][lec["t"] > 2])*1000:7.3f} mm   (max {lec["pos_max"]*1000:.1f} mm)')
print(f'joints moved ||q(6s) - q_home|| = {lec["drift"]:.2f} rad,   peak |tau| = {lec["tau_max"]:.0f} Nm')
check('Q3 OSC tracks the circle while it lasts (< 2 mm, 0.5-2 s)', np.median(lec['pe'][early]) < 2e-3, True)
plot_run(lec, 'Q3: OSC as written, 7-joint arm')

### Watch it

The RGB frame is the commanded pose; the yellow ball is the tool. (Rendering is
*given* and not needed to answer any question. On Linux without a display, set
`MUJOCO_GL=egl`; if it still will not cooperate, the numbers above are enough.)

In [ ]:
import sys; sys.path.append('../ForteArm_Model')   # so we can import the visualization code
from forte_viz import ForteViz, show_video, show_filmstrip
viz = ForteViz()
cam = viz.camera(zoom=1.5)

def render(frames):
    return [viz.frame(q, tgt, cam=cam) for q, tgt in frames]

frames_lec = render(lec['frames'])
show_filmstrip(frames_lec, n=6, cols=3, title='OSC as written: the arm drifts in its null space until a joint hits its limit')

In [ ]:
show_video(frames_lec, fps=25, path='osc_no_nullspace.mp4')   # half speed

**(Your answer for Q3d)**

*write here*

---
# Q4 [15 pts] The null space: using the seventh joint

$J$ is $6\times7$, so some joint motions do not move the tool at all. OSC gives you a
second control channel for them: add a torque $N^{\top}\tau_0$, where

$$\bar J = M^{-1}J^{\top}\Lambda, \qquad N = I - \bar J J .$$

$\bar J$ is the *dynamically consistent* pseudo-inverse of $J$ (it appears in the
derivation from class), and $N$ is the null-space projector built from it. $N$ acts
in **joint space**: $JN = 0$, so a joint velocity $\dot q = N v$ never moves the tool,
whatever $v$ is. Torques are the dual of velocities ($\tau = J^{\top}F$ is the dual of
$\dot x = J\dot q$), so a torque is projected with the **transpose**, $N^{\top}$.

**(a)** Complete `null_space_projector`, which returns $N$. The self-test checks three
identities at a random state:

1. $JN = 0$: $N$ spans the joint motions that do not move the tool.
2. $JM^{-1}N^{\top} = 0$. A torque $\tau$ produces the task acceleration $JM^{-1}\tau$, so
   this says $N^{\top}\tau_0$ cannot accelerate the tool, whatever $\tau_0$ is.
3. $J^{\top}(\mu + \rho + \Lambda\dot J\dot q) + N^{\top}(C + G) = C + G$. The OSC law of Q3
   compensates only the first term. The rest of the arm's own Coriolis and gravity
   torque, $N^{\top}(C+G)$, acts entirely in the null space, and nothing in Q3 opposes it.

**(b)** Complete `null_space_torque`. With `posture=False` it returns $N^{\top}(C+G)$, which
finishes the compensation. With `posture=True` it also holds the arm near a preferred
posture $q_{post}$ with a joint-space PD shaped by the inertia:

$$\tau_0 = M\big(K_{p,n}(q_{post}-q) - K_{d,n}\dot q\big) + C + G .$$

**(c)** The cell runs the circle for 12 s three ways: as in Q3, with $N^{\top}(C+G)$ added,
and with the posture task. Explain the three rows of the table. Why is $N^{\top}(C+G)$ alone
not enough? Why doesn't the posture task disturb the tool?

In [ ]:
def null_space_projector():
    """Dynamically consistent null-space projector N = I - Jbar J (NV x NV), in joint space."""
    M = get_M(); Minv = np.linalg.inv(M); J = get_J()
    # TODO Q4a: Lam = (J M^-1 J^T)^-1,  Jbar = M^-1 J^T Lam,  N = I - Jbar J  (joint space).
    raise NotImplementedError("Q4: null_space_projector")
    return N

def null_space_torque(q_post=None, Kp_n=25.0, Kd_n=10.0, posture=True):
    """N^T tau_0, with tau_0 = M (Kp_n (q_post - q) - Kd_n qdot) + C + G
    (posture=True), or tau_0 = C + G (posture=False)."""
    q_post = q_home if q_post is None else q_post
    # TODO Q4b: tau_0 = C + G, plus M (Kp_n (q_post - q) - Kd_n qdot) if posture.
    #           Return the projected torque N^T tau_0 (use null_space_projector().T).
    raise NotImplementedError("Q4: null_space_torque")


In [ ]:
# (a) the three identities, at a random state
rng = np.random.default_rng(4)
reset(); d.qpos[:] = np.clip(q_home + rng.normal(0, 0.3, NV), Q_LO, Q_HI)
d.qvel[:] = rng.normal(size=NV) * 0.5; mujoco.mj_forward(m, d)
M, C, G, J = get_M(), get_C(), get_G(), get_J()
N = null_space_projector()
Lam, mu, rho = task_dynamics()
check('Q4 J N = 0', J @ N, np.zeros((6, NV)), tol=1e-9)
check('Q4 J M^-1 N^T = 0', J @ np.linalg.inv(M) @ N.T, np.zeros((6, NV)), tol=1e-9)
check('Q4 J^T (mu + rho + Lambda Jdot qdot) + N^T (C + G) = C + G',
      J.T @ (mu + rho + Lam @ get_Jdot_qdot()) + N.T @ (C + G), C + G, tol=1e-9)
reset()
print(f'\nat ready_pose, at rest: |N^T G| = {np.linalg.norm(null_space_projector().T @ get_G()):.4f} Nm'
      f'   (|G| = {np.linalg.norm(get_G()):.3f} Nm)')

# (c) three versions of the circle, 12 s each
runs4 = {'OSC as in Q3':          dict(),
         '+ N^T (C + G)':         dict(null_space=lambda: null_space_torque(posture=False)),
         '+ N^T (posture + C + G)': dict(null_space=lambda: null_space_torque(posture=True))}
out4 = {k: run_circle(T=12.0, stride=15, **kw) for k, kw in runs4.items()}   # keep a frame every 30 ms
print(f'\n{"":24s} {"median":>8s} {"max":>8s}   {"||q - q_home||":>14s}  {"closest limit":>13s}')
print(f'{"":24s} {"[mm]":>8s} {"[mm]":>8s}   {"at 12 s [rad]":>14s}  {"[rad]":>13s}')
for k, r in out4.items():
    print(f'{k:24s} {r["pos_med"]*1000:8.3f} {r["pos_max"]*1000:8.2f}   {r["drift"]:14.3f}  {r["margin"]:13.3f}')
full = out4['+ N^T (posture + C + G)']
check('Q4 OSC + posture tracks the circle below 1 mm', full['pos_max'] < 1e-3, True)
plot_run(full, 'Q4: OSC + null-space posture')

In [ ]:
# the three versions of (c), side by side: four frames evenly spaced over 12 s each
from IPython.display import display, Markdown
videos4 = {}
for k, (name, r) in enumerate(out4.items()):
    videos4[name] = render(r['frames'])
    show_filmstrip(videos4[name], n=4, cols=4, title=f'{name}  (t = 0 to 12 s)')

In [ ]:
# ... and as videos (12 s each, real time)
for k, (name, frames) in enumerate(videos4.items()):
    display(Markdown(f'**{name}**'))
    display(show_video(frames, fps=33, path=f'osc_q4_case{k + 1}.mp4', width=600))

**(Your answer for Q4c)**

*write here*

---
# Q5 [20 pts] What each term does

Every term in the control law is there for a reason. This question switches them off,
in two groups. Every run tracks the circle for 4 s with the null-space posture task on,
except the $J^{\top}$ PD row (see below).

### Part 1 — the model terms: gravity and inertia (circle at 0.5 Hz)

| run | what changes |
|---|---|
| full OSC | — |
| no $\rho$ | `use_rho=False` (gravity on the task not compensated) |
| $J^{\top}$ PD | `use_Lambda=False`: $\tau = J^{\top}\ddot x_d + G$, same gains |

The $J^{\top}$ PD row is the "Cartesian PD" controller from class. It has no $\Lambda$,
so it cannot use the null-space projector and runs without it.

**(a)** Run the cell. It produces the table (median and maximum position error, median
orientation error, peak $|\tau|$, and % of timesteps that hit a torque limit) and
plots the error traces.

**(b)** Explain each row. Dropping $\rho$ costs a factor of about 50, while dropping
$\Lambda$ costs a factor of several hundred and saturates the motors. Why is $\Lambda$
the more important of the two? What is $J^{\top}\ddot x_d$ physically doing wrong?
Would raising the gains fix it? Argue from the eigenvalues of $\Lambda$ you found in Q1c.


In [ ]:
posture = lambda: null_space_torque(posture=True)
runs5 = {'full OSC':          dict(null_space=posture),
         'no rho':            dict(null_space=posture, use_rho=False),
         'J^T PD':            dict(use_Lambda=False)}
out5 = {k: run_circle(T=4.0, stride=15, **kw) for k, kw in runs5.items()}   # keep a frame every 30 ms

print(f'{"controller":20s} {"pos med":>9s} {"pos max":>9s} {"ori med":>9s} {"max|tau|":>9s} {"sat":>6s}')
print(f'{"":20s} {"[mm]":>9s} {"[mm]":>9s} {"[rad]":>9s} {"[Nm]":>9s} {"[%]":>6s}')
print('-' * 66)
for k, r in out5.items():
    if r is None:
        print(f'{k:20s}  diverged'); continue
    print(f'{k:20s} {r["pos_med"]*1000:9.3f} {r["pos_max"]*1000:9.2f} {r["ori_med"]:9.4f} '
          f'{r["tau_max"]:9.1f} {r["sat"]:6.1f}')

plt.figure(figsize=(8, 4.5))
for k, r in out5.items():
    if r is not None:
        plt.semilogy(r['t'], r['pe'] * 1000 + 1e-4, label=k, lw=1)
plt.xlabel('t [s]'); plt.ylabel('tool position error [mm]'); plt.legend(fontsize=8)
plt.grid(alpha=0.3, which='both'); plt.title('Q5 part 1: gravity and inertia'); plt.show()

In [ ]:
# replay part 1: 4 s each, real time
from IPython.display import display, Markdown
for k, (name, r) in enumerate(out5.items()):
    if r is None:
        continue
    display(Markdown(f'**{name}** (circle at {FREQ:g} Hz)'))
    display(show_video(render(r['frames']), fps=33, path=f'osc_q5_part1_{k + 1}.mp4', width=600))

**(Your answer for Q5b)**

*write here*

### Part 2 — the motion terms: how they scale with speed

$\mu$ (Coriolis and centrifugal forces, and $\dot J\dot q$) and the feedforward
$\ddot x_{ff}$ exist only because things move: with the arm at rest both are zero. At
0.5 Hz, leaving them out costs little. This part finds out how quickly that changes as
the circle speeds up. The radius stays $r = 10$ cm, so at frequency $f$ the tool moves at
$2\pi f r$ and its centripetal acceleration is $(2\pi f)^2 r$.

| run | what changes |
|---|---|
| full OSC | — |
| no $\mu$ | `use_mu=False` |
| no $\ddot x_{ff}$ | `use_ff=False` |

**(c)** Write the sweep. For each frequency in `SPEEDS` $= (0.25, 0.5, 1, 2)$ Hz and each
run in `VARIANTS`, call `run_circle(T=4.0, null_space=posture, **flags)` and store the
result as `speed[name][f]`. `circle(t)` reads the global `FREQ`, so set `FREQ = f` before
the runs at that frequency. Do it at the top level of the cell, not inside a function
(where you would need `global FREQ`). The cell restores `FREQ = 0.5` afterwards. It then
prints the median position and orientation errors, and plots them against $f$ on
log–log axes with reference slopes $\propto f$ and $\propto f^2$.

**(d)** Answer these:

1. Each frequency is double the one before. By what factor does each error grow per
   doubling? Which errors grow $\propto f$ and which $\propto f^2$?
2. **Full OSC.** Why does its error grow $\propto f$? *Hint:* compare it with
   $2\pi f r\,\Delta t$, $\Delta t = 2$ ms (see also Q7).
3. **No $\ddot x_{ff}$.** Without feedforward, $\ddot x = K_p e + K_d\dot e$ with
   $e = x_{ref} - x$, so the error obeys $\ddot e + K_d\dot e + K_p e = \ddot x_{ref}$.
   For a circle, $\ddot x_{ref}$ has amplitude $r\omega^2$ ($\omega = 2\pi f$). Show that the
   error amplitude is
   $$|e| = \frac{r\omega^2}{\sqrt{(K_p - \omega^2)^2 + (K_d\omega)^2}},$$
   evaluate it at the four frequencies ($K_p = 400$, $K_d = 40$) and compare with the
   table. Why does the growth slow down slightly at 2 Hz?
4. **No $\mu$.** Why does leaving out $\mu$ show up more in orientation than in position,
   and why does it grow $\propto f^2$? Name a robot motion where you could not leave
   $\mu$ out.

In [ ]:
SPEEDS = (0.25, 0.5, 1.0, 2.0)                    # circle frequency [Hz]; each one doubles the last
VARIANTS = {'full OSC':  dict(),
            'no mu':     dict(use_mu=False),
            'no xdd_ff': dict(use_ff=False)}
speed = {name: {} for name in VARIANTS}           # speed[name][f] = result of run_circle
# TODO Q5c: for each f in SPEEDS: set FREQ = f (circle(t) reads it), then for each
#   name, flags in VARIANTS.items(): speed[name][f] = run_circle(T=4.0, null_space=posture, **flags)
raise NotImplementedError("Q5: speed sweep")
FREQ = 0.5                                        # back to the default for the rest of the notebook

assert all(len(speed[n]) == len(SPEEDS) for n in VARIANTS), 'fill speed[name][f] for every run and frequency'
print(f'{"f [Hz]":>16s} ' + ''.join(f'{f:>10.2f}' for f in SPEEDS))
print(f'{"tool speed [m/s]":>16s} ' + ''.join(f'{2*np.pi*f*RADIUS:>10.3f}' for f in SPEEDS))
print('\nmedian position error [mm]')
for n in VARIANTS:
    print(f'{n:>16s} ' + ''.join(f'{speed[n][f]["pos_med"]*1000:>10.3f}' for f in SPEEDS))
print('\nmedian orientation error [rad]')
for n in VARIANTS:
    print(f'{n:>16s} ' + ''.join(f'{speed[n][f]["ori_med"]:>10.4f}' for f in SPEEDS))
print('\npeak |tau| [Nm] (full OSC): ' + '  '.join(f'{speed["full OSC"][f]["tau_max"]:.1f}' for f in SPEEDS)
      + f'   (limits {TAU_LIM})')
check('Q5 full OSC stays below 3 mm up to 2 Hz', max(speed['full OSC'][f]['pos_med'] for f in SPEEDS) < 3e-3, True)

fs = np.array(SPEEDS)
fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))
for n in VARIANTS:
    ax[0].loglog(fs, [speed[n][f]['pos_med'] * 1000 for f in SPEEDS], 'o-', label=n)
    ax[1].loglog(fs, [speed[n][f]['ori_med'] for f in SPEEDS], 'o-', label=n)
for a, key, scale in ((ax[0], 'pos_med', 1000), (ax[1], 'ori_med', 1)):
    y0 = min(speed[n][fs[0]][key] for n in VARIANTS) * scale
    a.loglog(fs, y0 * (fs / fs[0]), 'k:', lw=1, label=r'$\propto f$')
    a.loglog(fs, y0 * (fs / fs[0])**2, 'k--', lw=1, label=r'$\propto f^2$')
    a.set_xlabel('circle frequency f [Hz]'); a.grid(alpha=0.3, which='both'); a.legend(fontsize=8)
    a.set_xticks(fs); a.set_xticklabels([f'{f:g}' for f in fs]); a.minorticks_off()
ax[0].set_ylabel('median position error [mm]'); ax[1].set_ylabel('median orientation error [rad]')
ax[0].set_title('Q5 part 2: position'); ax[1].set_title('Q5 part 2: orientation')
plt.tight_layout(); plt.show()

In [ ]:
# replay part 2 at the highest speed only: 4 s each, real time
FREQ = max(SPEEDS)
fast = {name: run_circle(T=4.0, stride=15, null_space=posture, **flags) for name, flags in VARIANTS.items()}
FREQ = 0.5                                        # back to the default
for k, (name, r) in enumerate(fast.items()):
    display(Markdown(f'**{name}** (circle at {max(SPEEDS):g} Hz, tool error {r["pos_med"]*1000:.1f} mm, '
                     f'orientation error {r["ori_med"]:.3f} rad)'))
    display(show_video(render(r['frames']), fps=33, path=f'osc_q5_part2_{k + 1}.mp4', width=600))

**(Your answer for Q5d)**

*write here*

---
# Q6 [20 pts] Gain tuning: the tool as a mass-spring-damper

Substitute $F = \Lambda\ddot x_d + \mu + \rho$ into $\Lambda\ddot x + \mu + \rho = F$:
$\Lambda$, $\mu$ and $\rho$ cancel, and what is left is $\ddot x = \ddot x_d$. For a
fixed target ($\dot x_{ref} = \ddot x_{ff} = 0$) every direction of the tool error
obeys

$$\ddot e + K_d\,\dot e + K_p\,e = 0,$$

a **unit** mass on a spring $K_p$ and a damper $K_d$ — the second-order system from
the feedback-control lecture, with $\omega_n = \sqrt{K_p}$ and
$K_d = 2\zeta\sqrt{K_p}$.

**(a)** Complete `kd_for`, which returns the damping gain for a given $K_p$ and
damping ratio $\zeta$.

**(b)** Measure the step response. `step_response(axis, height, zeta)` (given) commands
a step of `height` meters along one world axis (0 = $x$, 1 = $y$, 2 = $z$) with
$K_p = 400$ ($\omega_n = 20$ rad/s), and returns the time `t` and the **normalized**
displacement `y` along that axis: $y = 0$ at the start, $y = 1$ at the target.

1. Complete `overshoot_settling(t, y, band=0.02)`, which returns two numbers:
   * **overshoot** in percent: how far the response goes past the target,
     $100\cdot\max\big(0,\ \max_k y_k - 1\big)$;
   * **settling time** in seconds: the time after which the response stays within the
     band, $|y - 1| \le$ `band`, until the end of the record. Use the first sample after
     the last one outside the band. Return `0.0` if the response is never outside the
     band, and `np.inf` if the last sample is still outside it (it has not settled).

   The test cell checks your function on made-up responses whose answers are known.
2. Write the sweep: for each $\zeta$ in `ZETAS` $= (0.2, 0.5, 0.7, 1, 2)$ and each axis
   0, 1, 2, run `step_response(axis, 0.03, zeta)` (a 3 cm step) and store
   `results[zeta][axis] = overshoot_settling(t, y)`. Also keep the $z$ response
   (axis 2) as `curves[zeta] = (t, y)` for the plot. The cell then prints the table and
   plots the $z$ responses.

**(c)** Answer these:

1. Describe the trend: as $\zeta$ increases, what happens to the overshoot and to the
   settling time? Why are the $x$, $y$ and $z$ responses the same, even though the arm is
   much heavier in one of those directions (see $\Lambda$ below)?
2. $\zeta = 2$ settles more slowly than $\zeta = 1$, although it has more damping.
   Explain why, in terms of the spring and the damper, and say which $\zeta$ you would
   choose for this arm.
3. The cell prints the translational block of $\Lambda$. Suppose you dropped $\Lambda$
   (the $J^{\top}$ PD of Q5) but kept the gains designed for $\zeta = 1$. Treating each
   direction as an independent mass $m_i = \Lambda_{ii}$, what damping ratio would
   the $x$ and $z$ directions actually get? (Recall the critical damping
   $b = 2\sqrt{km}$ from class.)

In [ ]:
def kd_for(kp, zeta):
    """Damping gain for a unit mass with stiffness kp and damping ratio zeta."""
    # TODO Q6a: damping gain for a unit mass with stiffness kp and damping ratio zeta.
    raise NotImplementedError("Q6: kd_for")

def step_response(axis, height, zeta, kp=400.0, T=1.5):
    """Step the tool target by `height` along world axis `axis` (0=x, 1=y, 2=z).
    Returns time and the normalized displacement (1 = reached the target)."""
    reset()
    Kp = np.array([100, 100, 100, kp, kp, kp], float)
    Kd = np.r_[kd_for(100.0, 1.0) * np.ones(3), kd_for(kp, zeta) * np.ones(3)]
    target = p_home.copy(); target[axis] += height
    ys = []
    for k in range(int(T / DT)):
        tau = osc_task(target, R_home, np.zeros(6), np.zeros(6), Kp, Kd) + null_space_torque()
        d.ctrl[:] = np.clip(tau, -TAU_LIM, TAU_LIM)
        mujoco.mj_step(m, d)
        ys.append((tool_pose()[0][axis] - p_home[axis]) / height)
    return np.arange(len(ys)) * DT, np.array(ys)

def overshoot_settling(t, y, band=0.02):
    """Overshoot [%] and settling time [s] of a normalized step response y(t) (target = 1)."""
    # TODO Q6b-1: overshoot = 100 * max(0, max(y) - 1).
    #   settling = time of the first sample after the LAST sample with |y - 1| > band;
    #   0.0 if no sample is outside the band, np.inf if the last sample is outside.
    raise NotImplementedError("Q6: overshoot_settling")
    return overshoot, settling

check('Q6 kd_for(400, 1) = 40', kd_for(400.0, 1.0), 40.0);

In [ ]:
# --- self test of overshoot_settling on made-up responses with known answers ---
t_ = np.arange(0, 2.0, 0.002)
os_, ts_ = overshoot_settling(t_, 1 - np.exp(-10 * t_))          # no overshoot, settles at ln(50)/10
check('Q6 smooth rise: no overshoot', os_, 0.0)
check('Q6 smooth rise: settles at ln(50)/10 = 0.391 s', ts_, np.log(50) / 10, tol=3e-3)
y_ = np.interp(t_, [0, 0.2, 0.5, 2.0], [0, 1.3, 1.0, 1.0])       # peaks at 1.3, back inside +2% at t = 0.48
os_, ts_ = overshoot_settling(t_, y_)
check('Q6 peak at 1.3: overshoot = 30 %', os_, 30.0, tol=1e-6)
check('Q6 peak at 1.3: settles at 0.48 s', ts_, 0.48, tol=3e-3)
check('Q6 already at the target: settling time 0', overshoot_settling(t_, np.ones_like(t_)), (0.0, 0.0))
check('Q6 never reaches the target: settling time inf', np.isinf(overshoot_settling(t_, 0.5 * np.ones_like(t_))[1]), True);

In [ ]:
ZETAS = (0.2, 0.5, 0.7, 1.0, 2.0)
results = {z: {} for z in ZETAS}   # results[zeta][axis] = (overshoot %, settling time s)
curves = {}                        # curves[zeta] = (t, y) of the z-axis response, for the plot
# TODO Q6b-2: for each zeta in ZETAS and each axis 0, 1, 2: t, y = step_response(axis, 0.03, zeta),
#   results[zeta][axis] = overshoot_settling(t, y); keep curves[zeta] = (t, y) for axis 2.
raise NotImplementedError("Q6: step-response sweep")

print(f'{"zeta":>5s}   ' + '   '.join(f'{a}: OS [%]  Ts [s]' for a in 'xyz'))
for z in ZETAS:
    print(f'{z:5.1f}   ' + '   '.join(f'{results[z][a][0]:9.1f}  {results[z][a][1]:6.3f}' for a in range(3)))

plt.figure(figsize=(8, 4.5))
for z, (t, y) in curves.items():
    plt.plot(t, y, label=f'zeta = {z}')
plt.axhline(1, color='k', lw=0.8)
plt.xlabel('t [s]'); plt.ylabel('normalized z displacement'); plt.legend(); plt.grid(alpha=0.3)
plt.title('Q6: 3 cm step in z, Kp = 400'); plt.show()

reset(); Lam, _, _ = task_dynamics()
print('translational block of Lambda at ready_pose [kg]:'); print(Lam[3:, 3:])

**(Your answer for Q6c)**

*write here*

---
# Q7 [BONUS 10 pts] Gains and the control rate

By Q6, $K_p$ sets the bandwidth $\omega_n = \sqrt{K_p}$, so it is tempting to raise
it until the error disappears. But the controller is digital: it computes a torque,
holds it until the next update, and only then sees the result.

**(a)** Run the cell. It sweeps the translational $K_p$ from $10^2$ to $10^5$ (with
$\zeta = 1$) at four control rates: 500 Hz (every simulation step), 100 Hz, 50 Hz and
25 Hz. It reports the median circle-tracking error and marks runs that went
unstable.

**(b)** Answer these:

1. At 500 Hz the error stops improving at about $0.63$ mm. What sets that floor?
   *Hint:* the reference moves at $2\pi \cdot 0.5 \cdot 0.10$ m/s ($\approx 0.3142$ m/s).
2. At lower rates the loop goes unstable above some $K_p$, and the threshold drops
   with the rate. Convert the thresholds to $f_n = \sqrt{K_p}/2\pi$ in Hz, compare
   them with the control rate, and state a rule of thumb. What $K_p$ would you
   allow on a robot whose controller runs at 200 Hz?

In [ ]:
RATES = [(1, '500 Hz'), (5, '100 Hz'), (10, '50 Hz'), (20, '25 Hz')]
print('median tracking error [mm]; "unstable" = diverged or error above 50 mm')
print(f'{"Kp":>8s} {"f_n [Hz]":>9s} ' + ' '.join(f'{lab:>10s}' for _, lab in RATES))
print('-' * 62)
for kp in (100., 400., 1600., 6400., 25600., 102400.):
    Kp = np.array([100, 100, 100, kp, kp, kp]); Kd = 2 * np.sqrt(Kp)
    row = []
    for dec, _ in RATES:
        r = run_circle(T=3.0, Kp=Kp, Kd=Kd, decim=dec, null_space=posture)
        row.append('unstable' if (r is None or r['pos_med'] > 0.05) else f'{r["pos_med"]*1000:.3f}')
    print(f'{kp:8.0f} {np.sqrt(kp)/(2*np.pi):9.1f} ' + ' '.join(f'{c:>10s}' for c in row))

**(Your answer for Q7b)**

*write here*

---
## Submission
Run all cells top to bottom — every `check(...)` should print **PASS** — then
export to PDF and upload it. Written answers are required for
**Q1c, Q2c, Q3d, Q4c, Q5b, Q5d** and **Q6c**, plus **Q7b** if you do the bonus.